# Inpaint Backgrounds

## Setup

In [ ]:
import sys
from pathlib import Path
from PIL import Image
import random

# Add the project root to sys.path to allow imports from avm package
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from avm.utils import load_info_records
from avm.inpaint import (
    MODELS,
    detect_device,
    get_prompt,
    build_model,
    inpaint_background,
    get_model_config_defaults
)

## Constants

In [5]:
IMAGES_DIR = '../data/bg1k_imgs/'  # Contains images named <ID>.png e.g., 0.png, 1.png, ...
MASKS_DIR = '../data/bg1k_masks/'  # Contains masks named <ID>_mask.png e.g., 0_mask.png
INFO_TXT_PATH = '../data/bg1k_info.txt'  # Each line: <image name>\t<category>
MODEL_ID = MODELS['stable-diffusion-xl-1.0-inpainting-0.1']
OUT_IMAGES_DIR = f'../data/bg1k_out_imgs/{MODEL_ID}/'  # Stores output images named <ID>.png

SEED = 42
PROMPT_OVERRIDE = ''  # If non-empty, overrides category prompt for all images

# Config overrides
CONFIG_OVERRIDES: dict[str, int | float] = {
    # Example overrides:
    # "num_inference_steps": 30,
}
MAX_RECORDS: int | None = 20  # None means no limit
OVERWRITE = False
DEVICE = None  # None -> auto detect via detect_device()

## Load Records

In [6]:
records = list(load_info_records(INFO_TXT_PATH))
if MAX_RECORDS is not None:
    records = random.Random(SEED).sample(records, min(MAX_RECORDS, len(records)))

print(f'Loaded {len(records)} records.')
print(f'Example records:')
for i, record in enumerate(records[:3]):
    print(f'{record}')

Loaded 20 records.
Example records:
{'image_id': '654', 'image_name': '654.png', 'category': 'Warm milk sterilization', 'mask_image_name': '654_mask.png'}
{'image_id': '114', 'image_name': '114.png', 'category': 'keyboard', 'mask_image_name': '114_mask.png'}
{'image_id': '25', 'image_name': '25.png', 'category': 'fabric sofa', 'mask_image_name': '25_mask.png'}


## Load Model

In [8]:
device = DEVICE or detect_device()
model = build_model(MODEL_ID, device)
model

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: /Users/nginyc/.cache/huggingface/hub/models--diffusers--stable-diffusion-xl-1.0-inpainting-0.1/snapshots/115134f363124c53c7d878647567d04daf26e41e/text_encoder_2
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /Users/nginyc/.cache/huggingface/hub/models--diffusers--stable-diffusion-xl-1.0-inpainting-0.1/snapshots/115134f363124c53c7d878647567d04daf26e41e/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Generate

In [9]:
config = {
    **get_model_config_defaults(MODEL_ID),
    **CONFIG_OVERRIDES,
}
print(f'Config: {config}')

Config: {'strength': 1.0, 'guidance_scale': 7.5, 'num_inference_steps': 50}


In [10]:
success = 0
skipped = 0
failed = 0

out_dir = Path(OUT_IMAGES_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

try:

    for idx, record in enumerate(records, start=1):
        image_name = record['image_name']
        mask_image_name = record['mask_image_name']
        category = record['category']
        image_path = Path(IMAGES_DIR) / image_name
        mask_image_path = Path(MASKS_DIR) / mask_image_name
        out_path = out_dir / image_name

        if out_path.exists() and not OVERWRITE:
            skipped += 1
            print(f'[{idx}/{len(records)}] Skipping existing output: {out_path.name}')
            continue

        category_prompt = get_prompt(category.strip())
        rendered_prompt = PROMPT_OVERRIDE.strip() or category_prompt

        print(f'[{idx}/{len(records)}] Processing {record["image_id"]}.png...')
        input_image = Image.open(image_path)
        mask_image = Image.open(mask_image_path)

        output_image = inpaint_background(
            model,
            input_image,
            mask_image,
            rendered_prompt,
            seed=SEED,
            config=config
        )

        output_image.save(out_path)
        success += 1
        print(f'[{idx}/{len(records)}] Saved: {out_path}')
except Exception as e:
    print(f'Error during processing: {e}')
finally:
    print(f'Finished processing records: {success} succeeded, {failed} failed, {skipped} skipped.')

[1/20] Processing 654.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[1/20] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/654.png
[2/20] Processing 114.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[2/20] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/114.png
[3/20] Skipping existing output: 25.png
[4/20] Processing 759.png...


...

[14/20] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/558.png
[15/20] Skipping existing output: 89.png
[16/20] Processing 604.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[16/20] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/604.png
[17/20] Processing 432.png...


  0%|          | 0/50 [00:00<?, ?it/s]

[17/20] Saved: ../data/bg1k_out_imgs/diffusers/stable-diffusion-xl-1.0-inpainting-0.1/432.png
[18/20] Skipping existing output: 32.png
[19/20] Skipping existing output: 30.png
[20/20] Skipping existing output: 95.png
Finished processing records: 15 succeeded, 0 failed, 5 skipped.
